# Data preparation for beech phenology

This notebook uses the included fixed sample of 50 beech (`Fagus sylvatica`) pixels and derives start-of-season (SOS) metrics from `NDVI`, `EVI`, and `IRECI`. The workflow represents a small public reproducibility example for the first data-preparation step of the phenology analysis.

In [1]:
# Imports
from pathlib import Path, PurePath

import numpy as np
import pandas as pd

from smoothing import savitzky_golay_filtering

In [ ]:
# Settings
PATCH_NAME = "33TVL_2_1_0_1"
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data" if (WORK_DIR / "data").exists() else WORK_DIR
OUTPUT_DIR = WORK_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 50
RANDOM_STATE = 42
BEECH_CLASS = 2
YEARS_TO_KEEP = range(2018, 2022)

CLASSIFICATION_CSV = DATA_DIR / "beech_sample_50pixels.csv"

VEG_INDICES = ["NDVI", "EVI", "IRECI"]
tresh_sos = 0.5

# For the public GitHub sample, use the included smoothed time-series CSV.
SMOOTHED_TS_CSV = DATA_DIR / "beech_sample_50pixels_timeseries_smoothed.csv"
if not SMOOTHED_TS_CSV.exists():
    raise FileNotFoundError(f"Smoothed time-series CSV not found: {SMOOTHED_TS_CSV}")

sample_csv = OUTPUT_DIR / "beech_sample_50pixels.csv"
pheno_head_csv = OUTPUT_DIR / "pheno_df_head.csv"
sos_csv = OUTPUT_DIR / "beech_sample_50pixels_sos_ndvi_evi_ireci.csv"

CLASSIFICATION_CSV, SMOOTHED_TS_CSV, sample_csv, pheno_head_csv, sos_csv

(WindowsPath('c:/Users/apotocni/OneDrive - Univerza v Ljubljani/Documents/4_Articles/1_Članki_Q1/1_V delu/Phenology_research/data_and_code/github_publication - Copy/data/33TVL_2_1_0_1_beech_sample_50pixels.csv'),
 WindowsPath('c:/Users/apotocni/OneDrive - Univerza v Ljubljani/Documents/4_Articles/1_Članki_Q1/1_V delu/Phenology_research/data_and_code/github_publication - Copy/data/33TVL_2_1_0_1_beech_sample_50pixels_timeseries_smoothed.csv'),
 WindowsPath('c:/Users/apotocni/OneDrive - Univerza v Ljubljani/Documents/4_Articles/1_Članki_Q1/1_V delu/Phenology_research/data_and_code/github_publication - Copy/processed/33TVL_2_1_0_1_beech_sample_50pixels.csv'),
 WindowsPath('c:/Users/apotocni/OneDrive - Univerza v Ljubljani/Documents/4_Articles/1_Članki_Q1/1_V delu/Phenology_research/data_and_code/github_publication - Copy/processed/pheno_df_head.csv'),
 WindowsPath('c:/Users/apotocni/OneDrive - Univerza v Ljubljani/Documents/4_Articles/1_Članki_Q1/1_V delu/Phenology_research/data_and_code/g

In [3]:
def add_vegetation_indices(s2_df):
    s2_df = s2_df.copy()

    ireci_denominator = s2_df["B05"] / s2_df["B06"]
    s2_df["IRECI"] = np.where(
        (s2_df["B05"] != 0) & (s2_df["B06"] != 0) & (ireci_denominator != 0),
        (s2_df["B07"] - s2_df["B04"]) / ireci_denominator,
        np.nan,
    )

    evi_denominator = s2_df["B08"] + 6 * s2_df["B04"] - 7.5 * s2_df["B02"] + 1
    s2_df["EVI"] = np.where(
        evi_denominator != 0,
        2.5 * (s2_df["B08"] - s2_df["B04"]) / evi_denominator,
        np.nan,
    )

    return s2_df


def smooth_ts_7d_adjusted(group):
    group = group.sort_values("TIMESTAMP").copy()
    smoothed_data = (
        group.set_index("TIMESTAMP")[VEG_INDICES]
        .resample("7D")
        .mean()
        .apply(savitzky_golay_filtering)
        .resample("1D")
        .mean()
        .interpolate()
        .reset_index()
    )
    smoothed_data["P_ID"] = group["P_ID"].iloc[0]
    return smoothed_data


def calculate_sos(pheno_df, vi_columns=VEG_INDICES, sos_threshold=tresh_sos):
    rows = []

    for p_id, p_id_data in pheno_df.groupby("P_ID"):
        p_id_data = p_id_data.copy()
        p_id_data["TIMESTAMP"] = pd.to_datetime(p_id_data["TIMESTAMP"])
        p_id_data = p_id_data.set_index("TIMESTAMP").sort_index()

        for year, year_data in p_id_data.groupby(p_id_data.index.year):
            for vi_name in vi_columns:
                point = year_data[vi_name].dropna()
                if point.empty:
                    continue

                max_value = point.max()
                max_index = point.idxmax()

                left = point.loc[:max_index]
                if left.empty:
                    continue

                min_idx_left = left.idxmin()
                min_value_left = left.min()

                amplitude_sos = max_value - min_value_left
                if amplitude_sos <= 0:
                    continue

                sos_value = (amplitude_sos * sos_threshold) + min_value_left
                sos_idx = point.loc[min_idx_left:max_index].sub(sos_value).abs().idxmin()

                rows.append({
                    "P_ID": p_id,
                    "Column_Name": vi_name,
                    "YEAR": year,
                    "SOS_idx": sos_idx,
                    "SOS_DOY": sos_idx.dayofyear,
                })

    return pd.DataFrame(rows)

In [4]:
sample_pixels = pd.read_csv(CLASSIFICATION_CSV)
sample_pixels["P_ID"] = sample_pixels["P_ID"].astype("int64")
sample_pixels = sample_pixels.drop_duplicates(subset=["P_ID"]).sort_values("P_ID").reset_index(drop=True)

if len(sample_pixels) != SAMPLE_SIZE:
    print(f"Warning: expected {SAMPLE_SIZE} pixels, found {len(sample_pixels)}")

sample_pixels.to_csv(sample_csv, index=False)
sample_pids = sample_pixels["P_ID"].tolist()

print("Loaded fixed beech sample:", sample_pixels["P_ID"].nunique(), "unique P_ID values")
sample_pixels.head()

Loaded fixed beech sample: 50 unique P_ID values


,P_ID
0,40257804
1,40262355
2,40271215
3,40274709
4,40275837


In [5]:
# For the public GitHub sample, load the included smoothed time series directly.
pheno_df = pd.read_csv(SMOOTHED_TS_CSV)
pheno_df["P_ID"] = pheno_df["P_ID"].astype("int64")
pheno_df["TIMESTAMP"] = pd.to_datetime(pheno_df["TIMESTAMP"])
pheno_df[VEG_INDICES] = pheno_df[VEG_INDICES].astype("float32")
pheno_df = pheno_df.sort_values(["P_ID", "TIMESTAMP"]).reset_index(drop=True)

pheno_df_head = pheno_df.head()
pheno_df_head.to_csv(pheno_head_csv, index=False)

print("Loaded smoothed time series:", pheno_df.shape)
print("Unique P_ID values:", pheno_df["P_ID"].nunique())
print("Saved pheno_df.head():", pheno_head_csv)
pheno_df_head

Loaded smoothed time series: (87711, 5)
Unique P_ID values: 50
Saved pheno_df.head(): c:\Users\apotocni\OneDrive - Univerza v Ljubljani\Documents\4_Articles\1_Članki_Q1\1_V delu\Phenology_research\data_and_code\github_publication - Copy\processed\pheno_df_head.csv


,TIMESTAMP,NDVI,EVI,IRECI,P_ID
0,2017-03-12,0.430086,0.188929,0.074582,40257804
1,2017-03-13,0.429402,0.190474,0.075049,40257804
2,2017-03-14,0.428719,0.192020,0.075516,40257804
3,2017-03-15,0.428035,0.193565,0.075982,40257804
4,2017-03-16,0.427351,0.195111,0.076449,40257804


In [6]:
sos_df = calculate_sos(pheno_df)

result_df = sample_pixels.merge(sos_df, on="P_ID", how="inner")
result_df = result_df[result_df["YEAR"].isin(YEARS_TO_KEEP)].copy()
result_df = result_df.sort_values(["P_ID", "Column_Name", "YEAR"]).reset_index(drop=True)
result_df.to_csv(sos_csv, index=False)

print("Saved:", sos_csv)
print("Years retained in the result:", sorted(result_df["YEAR"].unique()))
print("Result:", result_df.shape)
print(result_df.groupby("Column_Name")["P_ID"].nunique())
result_df.head(15)

Saved: c:\Users\apotocni\OneDrive - Univerza v Ljubljani\Documents\4_Articles\1_Članki_Q1\1_V delu\Phenology_research\data_and_code\github_publication - Copy\processed\33TVL_2_1_0_1_beech_sample_50pixels_sos_ndvi_evi_ireci.csv
Years retained in the result: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Result: (600, 5)
Column_Name
EVI      50
IRECI    50
NDVI     50
Name: P_ID, dtype: int64


,P_ID,Column_Name,YEAR,SOS_idx,SOS_DOY
0,40257804,EVI,2018,2018-04-23,113
1,40257804,EVI,2019,2019-04-22,112
2,40257804,EVI,2020,2020-04-18,109
3,40257804,EVI,2021,2021-05-02,122
4,40257804,IRECI,2018,2018-04-27,117
5,40257804,IRECI,2019,2019-05-01,121
6,40257804,IRECI,2020,2020-04-27,118
7,40257804,IRECI,2021,2021-05-14,134
8,40257804,NDVI,2018,2018-04-16,106
9,40257804,NDVI,2019,2019-04-20,110
